# Chapter 2 Project — Titanic Survival Prediction

This project applies everything from chapter 2 on one real dataset — end to end.

**Goal:** Predict whether a passenger survived the Titanic disaster.

**What this covers:**
- EDA (2.6)
- Handling missing values (2.1)
- Handling outliers (2.2)
- Encoding categorical data (2.3)
- Feature engineering (2.5)
- Feature scaling (2.4)
- Training a baseline model and evaluating it

**Dataset:** Titanic passenger data — 891 rows, 12 columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
df = sns.load_dataset('titanic')
print("Shape:", df.shape)
df.head()

## Step 1 — EDA

Understand the data before touching it.

In [ ]:
df.info()

In [ ]:
# Missing values
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Class balance
print(df['survived'].value_counts(normalize=True).round(3) * 100)

In [ ]:
# Key groupby findings
print("Survival by sex:")
print(df.groupby('sex')['survived'].mean().round(2))

print("\nSurvival by pclass:")
print(df.groupby('pclass')['survived'].mean().round(2))

## Step 2 — Select & Clean Columns

Drop columns that are too messy or not useful for prediction.

In [ ]:
# Keep only useful columns
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()

print("Working with:", df.shape)
df.head()

## Step 3 — Handle Missing Values (2.1)

In [ ]:
# Age — fill with median (skewed distribution)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked — fill with mode (only 2 missing)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# Verify
print("Missing values remaining:")
print(df.isnull().sum())

## Step 4 — Handle Outliers (2.2)

In [ ]:
# Fare has extreme outliers — cap using IQR
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

print(f"Fare upper bound: {upper:.1f}")
print(f"Values above bound: {(df['fare'] > upper).sum()}")

df['fare'] = df['fare'].clip(upper=upper)
print(f"Fare max after capping: {df['fare'].max():.1f}")

## Step 5 — Feature Engineering (2.5)

In [ ]:
# family_size — travelling alone vs with family
df['family_size'] = df['sibsp'] + df['parch'] + 1

# is_alone — binary flag
df['is_alone'] = (df['family_size'] == 1).astype(int)

print("Survival rate by is_alone:")
print(df.groupby('is_alone')['survived'].mean().round(2))

df[['family_size', 'is_alone', 'survived']].head()

## Step 6 — Encode Categorical Columns (2.3)

In [ ]:
# sex — label encode (binary: male=1, female=0)
df['sex'] = LabelEncoder().fit_transform(df['sex'])

# embarked — one-hot encode (no natural order)
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

print("Columns after encoding:", df.columns.tolist())
df.head()

## Step 7 — Split, Scale, Train (2.4)

In [ ]:
X = df.drop(columns=['survived'])
y = df['survived']

# Split first — always
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale after split — fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Train size:", X_train.shape)
print("Test size: ", X_test.shape)

## Step 8 — Train a Baseline Model

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Step 9 — Reflect

Write your observations here:

**What the data needed:**
- Missing values in age (median fill) and embarked (mode fill)
- Fare had outliers — capped with IQR
- Sex and embarked needed encoding
- Created family_size and is_alone as new features

**Model performance:**
- Baseline logistic regression accuracy: ~?
- Strong features: sex, pclass, fare

**What could improve it:**
- Try other algorithms (chapter 4)
- Extract title from passenger name
- Handle class imbalance (chapter 7)

In [ ]:
# Feature importance — which features did the model rely on most?
importance = pd.Series(
    abs(model.coef_[0]),
    index=X.columns
).sort_values(ascending=False)

print("Feature importance (logistic regression coefficients):")
print(importance.round(3))